# Hybrid XGBoost + CNN Training V2

All modes use a **pretrained XGBoost** model loaded from `XGB_MODEL_PATH`.

| Mode | CNN input | Inference |
|---|---|---|
| `'prob_only'` | XGB prob (1 ch) | CNN output directly |
| `'prob_feat'` | features + XGB prob (C+1 ch) | CNN output directly |
| `'parallel'` | features only (C ch) | combine(CNN prob, XGB prob) |

For `patch_mode='full_padding'` the crop size is set automatically to `(min_H, min_W)`
across all samples; a random jitter shift is applied at dataset creation time.

## 1. Setup

In [7]:
import sys, os, socket
import torch
import importlib

# Two levels up: HybridTrainV2/ → NNsTorchV2/ → KIprojV2_Claude/ (project root)
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import NNsTorchV2
importlib.reload(NNsTorchV2)
import NNsTorchV2.HybridTrainV2 as htv2
importlib.reload(htv2)

from NNsTorchV2.HybridTrainV2 import HybridTrainingManager, build_hybrid_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def detect_system():
    h = socket.gethostname()
    if h.startswith('VDI0147'): return 'Thermo10'
    if h.startswith('NB'):      return 'Windows'
    return 'GPU'
SYSTEM = detect_system()
print(f'System: {SYSTEM}')

Device: cpu
System: GPU


## 2. Configuration

In [8]:
# ── Mode ────────────────────────────────────────────────────────────────────
MODE      = 'parallel'   # 'prob_only' | 'prob_feat' | 'parallel'| 'nn_only'
COMBINE   = 'mean'        # 'mean' | 'max'  — only used for 'parallel' mode-best prob or meansum
N_FILTERS = 32
INIT_W    = -10            #0 default#10 for 0.999, -10 for 0.001

# ── Data ─────────────────────────────────────────────────────────────────────
POWER_MODE  = '4kw_both'
SUBFOLDER   = 'Taris/Data_ML_V3'
MASK_TYPE   = 'alternative'
INVERT_MASK = True
DATA_REGIME = 'postprocessed'
PPT_PHASES  = 'all'
PPT_AMPS    = 6
DIRS        = [0, 1, 2, 3, 4, 5, 6]   # [] = all
FROZE_W     = 5
# ── Patch / image size ────────────────────────────────────────────────────────
PATCH_MODE  = 'full_padding'  # 'full_padding' (auto min H/W) | 'patches' | 'full'
PATCH_SIZE  = (64, 64)      # used for 'patches'; overridden for 'full_padding'

# ── Training ──────────────────────────────────────────────────────────────────
MODEL_NAME = 'cnn_se'
N_SPLITS   = 3
EPOCHS     = 3
PATIENCE   = 10
BATCH_SIZE = 4
LR         = 1e-2#3
LOSS_NAME  = 'tversky'   # see NNsTorch/losses.py
ALPHA      = 0.5        # Tversky FP penalty
BETA       = 0.5        # Tversky FN penalty

# ── XGBoost ───────────────────────────────────────────────────────────────────
XGB_MODEL_PATH = '../../Trees/Trees_database/XGB_setV3[all]_md30minch30gam0.1_lr0.05_n300_normalized_scale2.joblib'

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_URI = 'sqlite:////tmp/mlflow_experiments/mlflow.db'
print(f'MLflow UI: mlflow ui --backend-store-uri {MLFLOW_URI} --host 127.0.0.1 --port 5000')

MLflow UI: mlflow ui --backend-store-uri sqlite:////tmp/mlflow_experiments/mlflow.db --host 127.0.0.1 --port 5000


## 3. Initialise Manager

In [9]:
manager = HybridTrainingManager(
    model_name     = MODEL_NAME,
    sys            = SYSTEM,
    mode           = MODE,
    xgb_model_path = XGB_MODEL_PATH,
    power_mode     = POWER_MODE,
    subfolder_name = SUBFOLDER,
    patch_size     = PATCH_SIZE,
    patch_mode     = PATCH_MODE,
    initial_lr     = LR,
    loss_name      = LOSS_NAME,
    alpha          = ALPHA,
    beta           = BETA,
    mask_type      = MASK_TYPE,
    dirs           = DIRS,
    ppt_phases     = PPT_PHASES,
    ppt_amps       = PPT_AMPS,
    invert_mask    = INVERT_MASK,
    data_regime    = DATA_REGIME,
    combine        = COMBINE,
    mlflow_uri     = MLFLOW_URI,
    fusion_freeze_epochs=FROZE_W,
    save_chckpnt   = True,
    init_w         = INIT_W
)

# model_fn is called once per fold; n_raw_ch is stored by manager
def model_fn():
    return build_hybrid_model(MODE, manager.n_raw_ch, N_FILTERS,model_name=MODEL_NAME)

n_params = sum(p.numel() for p in model_fn().parameters())
print(f'Input shape : {manager.input_shape}')
print(f'Model params: {n_params:,}')

Device: cpu  |  mode: parallel  |  patch_mode: full_padding


/home/aaverin/miniforge3/envs/pytorch_env/lib/python3.11/pickle.py:1718: UserWarning: [11:15:41] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


Loaded XGBoost: XGB_setV3[all]_md30minch30gam0.1_lr0.05_n300_normalized_scale2.joblib
Output directory: cnn_se
MLflow tracking: sqlite:////tmp/mlflow_experiments/mlflow.db
Discovered 57 samples
Auto patch_size (min H/W): [346, 369]
n_raw_ch=82  CNN input: (82, 346, 369)  (mode=parallel)
Input shape : (82, 346, 369)
Model params: 38,273


## 4. Training

In [ ]:
fold_metrics, avg = manager.run_kfold(
    model_fn,
    n_splits   = N_SPLITS,
    batch_size = BATCH_SIZE,
    epochs     = EPOCHS,
    patience   = PATIENCE,
)


Fold 1/3
TRAIN: 38 — Counter({1: 17, 0: 12, 2: 9})
VAL:   19 — Counter({1: 9, 0: 6, 2: 4})


## 5. Results

In [6]:
names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'IoU']
print(f'Mode: {MODE}  combine: {COMBINE}  |  Model: {manager.versioned_name}')
print('-' * 60)
for i, fm in enumerate(fold_metrics):
    print(f'Fold {i+1}: ' + '  '.join(f'{n}={v:.4f}' for n, v in zip(names, fm)))
print('=' * 60)
print('Average:')
for n, v in zip(names, avg):
    print(f'  {n:12s}: {v:.4f}')
print(f'\nCheckpoints: {manager.ckpt_dir}')

Mode: prob_only  combine: mean  |  Model: hybrid_prob_only
------------------------------------------------------------
Fold 1: Loss=0.2064  Accuracy=0.9906  Precision=0.9423  Recall=0.9502  IoU=0.8978
Fold 2: Loss=0.1934  Accuracy=0.9884  Precision=0.9360  Recall=0.9354  IoU=0.8777
Fold 3: Loss=0.2954  Accuracy=0.9911  Precision=0.9523  Recall=0.9322  IoU=0.8891
Average:
  Loss        : 0.2317
  Accuracy    : 0.9901
  Precision   : 0.9436
  Recall      : 0.9393
  IoU         : 0.8882

Checkpoints: /home/aaverin/RZ-Dienste/hpc-user/aaverin/Python/KIprojV2_Claude/NNsTorch/HybridTrainV2/checkpoints/hybrid_prob_only/20260304-093711


## 6. Quick Inference Check (optional)

Load a saved fold model and visualise predictions on one sample.

In [ ]:
import matplotlib.pyplot as plt
from NNsTorch.HybridTrainV2.hybrid_utils import HybridPatchDataset

# ── Checkpoint ────────────────────────────────────────────────────────────────
FOLD_TO_CHECK = 1

# Manual: set path explicitly (works without manager)
CKPT_DIR  = '/home/aaverin/RZ-Dienste/hpc-user/aaverin/Python/KIprojV2_Claude/NNsTorch/HybridTrainV2/checkpoints/cnn_seprob_feat/20260306-083610'
CKPT_FILE = f'fold{FOLD_TO_CHECK}_best.pt'   # e.g. 'fold1_ep05.pt' or 'fold1_best.pt'
#NNsTorch/HybridTrainV2/checkpoints/cnn_seprob_feat/20260306-083610
# Auto (requires manager): uncomment to use
CKPT_DIR  = manager.ckpt_dir
CKPT_FILE = f'fold{FOLD_TO_CHECK}_best.pt'

ckpt_path = os.path.join(CKPT_DIR, CKPT_FILE)

# ── Pick samples: (sample_name, location_idx) ─────────────────────────────────
SAMPLES_TO_CHECK = [
    ('s1', 1),
    ('s1', 3),
    ('s3', 1),
    ('s3', 5),
    ('s5', 2),
    ('s5', 4),
]

# Reload best model from checkpoint
ckpt_model = model_fn().to(device)
ckpt_model.load_state_dict(
    torch.load(ckpt_path, map_location=device)['model_state_dict'])
ckpt_model.eval()

for sample in SAMPLES_TO_CHECK:
    check_ds = HybridPatchDataset(
        sample_indices = [sample],
        xgb_model      = manager.xgb_global,
        load_path      = manager.load_path,
        power_mode     = manager.power_mode,
        patch_size     = manager.patch_size,
        augment        = False,
        mask_type      = manager.mask_type,
        ppt_phases     = manager.ppt_phases,
        ppt_amps       = manager.ppt_amps,
        invert_mask    = manager.invert_mask,
        apply_jitter   = False,
        patch_mode     = manager.patch_mode,
        data_regime    = manager.data_regime,
    )
    data_t, xgb_t, mask_t = check_ds[0]

    with torch.no_grad():
        if MODE == 'prob_only':
            inp = xgb_t.unsqueeze(0).to(device)
        elif MODE == 'prob_feat':
            inp = torch.cat([data_t, xgb_t], dim=0).unsqueeze(0).to(device)
        else:  # parallel / nn_only
            inp = data_t.unsqueeze(0).to(device)
        cnn_prob = torch.sigmoid(ckpt_model(inp)).squeeze().cpu().numpy()

    xgb_np  = xgb_t.squeeze().numpy()
    #xgb_np = (xgb_np > 0.5).astype(int)
    mask_np = mask_t.numpy()

    if MODE == 'parallel':
        combined = (cnn_prob + xgb_np) / 2
        titles = ['XGB Prob', 'CNN Prob', f'Combined ({COMBINE})', 'Ground Truth']
        imgs   = [xgb_np, cnn_prob, combined, mask_np]
        cmaps  = ['hot', 'hot', 'hot', 'gray']
    else:
        titles = ['XGB Prob', 'CNN Output', 'Ground Truth']
        imgs   = [xgb_np, cnn_prob, mask_np]
        cmaps  = ['hot', 'hot', 'gray']

    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    for ax, img, title, cmap in zip(axes, imgs, titles, cmaps):
        ax.imshow(img, cmap=cmap); ax.set_title(title); ax.axis('off')
    plt.suptitle(f"sample={sample[0]}  loc={sample[1]}  |  mode={MODE}  |  ckpt={CKPT_FILE}")
    plt.tight_layout(); plt.show()